In [1]:
!pip install ultralytics

In [2]:
import ultralytics, inspect
import os
import yaml
from ultralytics.utils.loss import BboxLoss
print("ultralytics:", ultralytics.__version__)
print(inspect.getsource(BboxLoss.__init__))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
ultralytics: 8.4.132
    def __init__(self, reg_max: int = 16):
        """Initialize the BboxLoss module with regularization maximum and DFL settings."""
        super().__init__()
        self.dfl_loss = DFLoss(reg_max) if reg_max > 1 else None



In [3]:
data_dir = '/kaggle/input/datasets/muhammadibnerafiq/combination4/Combination4'


data_yaml = {
    "path": data_dir,  # root path of dataset
    "train": "images/train",   # relative to 'path'
    "val":   "images/valid",   # or 'images/val' if that is the folder
    "test": "images/test",   
    "names": {
        0: "pole",      
    }
}

save_path = "/kaggle/working/data_custom.yaml"

os.makedirs(os.path.dirname(save_path), exist_ok=True)
with open(save_path, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(f"data.yaml written to: {save_path}")


data.yaml written to: /kaggle/working/data_custom.yaml


In [4]:
# %%writefile /kaggle/working/compute_nwd_C.py
# # <paste the full file here>
# """
# Compute the NWD constant C from YOLO-format labels.

# Auto-detects image size from a sample image so you don't have to guess.

#     python compute_nwd_C.py --labels /path/to/labels/train \
#                             --images /path/to/images/train
# """
# import argparse, glob, os
# import numpy as np

# try:
#     from PIL import Image
#     HAVE_PIL = True
# except ImportError:
#     HAVE_PIL = False


# def detect_size(image_dir):
#     exts = ("*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff", "*.bmp")
#     files = []
#     for e in exts:
#         files += glob.glob(os.path.join(image_dir, e))
#     if not files:
#         return None, None, 0
#     sizes = []
#     for f in files[:50]:
#         with Image.open(f) as im:
#             sizes.append(im.size)          # (w, h)
#     uniq = set(sizes)
#     if len(uniq) > 1:
#         print(f"WARNING: {len(uniq)} distinct image sizes found. Using the most common.")
#     w, h = max(set(sizes), key=sizes.count)
#     return w, h, len(files)


# def load_boxes(label_dir):
#     ws, hs = [], []
#     files = glob.glob(os.path.join(label_dir, "*.txt"))
#     if not files:
#         raise SystemExit(f"No .txt label files in {label_dir}")
#     empty = 0
#     for f in files:
#         n = 0
#         with open(f) as fh:
#             for line in fh:
#                 p = line.split()
#                 if len(p) < 5:
#                     continue
#                 ws.append(float(p[3])); hs.append(float(p[4])); n += 1
#         if n == 0:
#             empty += 1
#     return np.array(ws), np.array(hs), len(files), empty


# def main():
#     ap = argparse.ArgumentParser()
#     ap.add_argument("--labels", required=True)
#     ap.add_argument("--images", default=None,
#                     help="image dir; used to auto-detect resolution")
#     ap.add_argument("--imgw", type=int, default=None)
#     ap.add_argument("--imgh", type=int, default=None)
#     a = ap.parse_args()

#     ws, hs, n_lbl, n_empty = load_boxes(a.labels)

#     imgw, imgh = a.imgw, a.imgh
#     if (imgw is None or imgh is None) and a.images and HAVE_PIL:
#         dw, dh, n_img = detect_size(a.images)
#         if dw:
#             imgw, imgh = imgw or dw, imgh or dh
#             print(f"Detected image size from {n_img} files: {dw} x {dh}")
#     if imgw is None or imgh is None:
#         raise SystemExit("Could not determine image size. Pass --imgw and --imgh.")

#     w_px, h_px = ws * imgw, hs * imgh
#     scale = np.sqrt(w_px * h_px)

#     print(f"\nLabel files: {n_lbl}   (empty: {n_empty})   Boxes: {len(ws)}")
#     print(f"Image size:  {imgw} x {imgh}\n")

#     print(f"{'':<20}{'mean':>9}{'median':>9}{'std':>9}{'min':>9}{'max':>9}")
#     print("-" * 65)
#     for name, arr in [("width  (px)", w_px), ("height (px)", h_px),
#                       ("sqrt(w*h) (px)", scale)]:
#         print(f"{name:<20}{arr.mean():>9.2f}{np.median(arr):>9.2f}"
#               f"{arr.std():>9.2f}{arr.min():>9.2f}{arr.max():>9.2f}")

#     C_img = scale.mean()
#     print("\n" + "=" * 65)
#     print(f"  C (image pixels) = {C_img:.3f}")
#     print("=" * 65)

#     print("\nIf boxes reach the loss divided by stride, matching C values:")
#     print(f"{'stride':>8}{'C':>12}")
#     for s in (8, 16, 32):
#         print(f"{s:>8}{C_img / s:>12.4f}")
#     print("\nThe [NWD] print on the first training batch tells you which")
#     print("units are actually in use. Trust that over this table.\n")

#     p10, p90 = np.percentile(scale, [10, 90])
#     print(f"Scale spread: p10={p10:.2f}  p90={p90:.2f}  (ratio {p90/max(p10,1e-9):.1f}x)")
#     print("  -> " + ("Wide. One global C is a compromise; evidence for a "
#                      "per-object adaptive blend."
#                      if p90 / max(p10, 1e-9) > 4 else
#                      "Narrow. One global C is well justified."))
#     tiny = (scale < 16).mean() * 100
#     print(f"\n{tiny:.1f}% of boxes are under 16px scale (the AI-TOD 'tiny' threshold).")


# if __name__ == "__main__":
#     main()

In [5]:
# !python /kaggle/working/compute_nwd_C.py \
#     --labels /kaggle/input/datasets/muhammadibnerafiq/combination4/Combination4/labels/train \
#     --images /kaggle/input/datasets/muhammadibnerafiq/combination4/Combination4/images/train

In [6]:
# %%writefile /kaggle/working/nwd_patch.py
# # <paste the full nwd_patch.py here>
# """
# NWD + IoU blended box loss for Ultralytics YOLO.

# Targets the 8.4.x BboxLoss signature (imgsz + stride args) and falls back
# to the older 7-arg signature automatically.

# Because `stride` is available, boxes are converted back to IMAGE PIXELS
# before computing NWD -- so C is in image pixels, exactly the number that
# compute_nwd_C.py reports.

#     import nwd_patch
#     nwd_patch.patch(C=18.0, ratio=0.5)
#     from ultralytics import YOLO
#     YOLO("yolo11n.pt").train(...)

# ratio = 0.0 -> pure IoU (baseline)
# ratio = 1.0 -> pure NWD
# """
# import torch
# import torch.nn.functional as F
# from ultralytics.utils import loss as ul_loss
# from ultralytics.utils.metrics import bbox_iou
# from ultralytics.utils.tal import bbox2dist

# _CFG = {"C": 18.0, "ratio": 0.5, "printed": False}
# _BASE = ul_loss.BboxLoss


# def nwd_similarity(box1, box2, C, eps=1e-7):
#     """NWD similarity in (0, 1]. Boxes xyxy, same units as C. 1.0 = identical."""
#     b1x1, b1y1, b1x2, b1y2 = box1.chunk(4, -1)
#     b2x1, b2y1, b2x2, b2y2 = box2.chunk(4, -1)

#     cx1, cy1 = (b1x1 + b1x2) / 2, (b1y1 + b1y2) / 2
#     cx2, cy2 = (b2x1 + b2x2) / 2, (b2y1 + b2y2) / 2
#     w1, h1 = (b1x2 - b1x1).clamp(min=eps), (b1y2 - b1y1).clamp(min=eps)
#     w2, h2 = (b2x2 - b2x1).clamp(min=eps), (b2y2 - b2y1).clamp(min=eps)

#     d2 = ((cx1 - cx2) ** 2 + (cy1 - cy2) ** 2
#           + ((w1 - w2) / 2) ** 2 + ((h1 - h2) / 2) ** 2)

#     return torch.exp(-torch.sqrt(d2.clamp(min=0) + eps) / C)


# def _blend(pb, tb, weight, target_scores_sum, stride_fg):
#     """(1-r)*IoU_loss + r*NWD_loss, NWD computed in image pixels."""
#     iou = bbox_iou(pb, tb, xywh=False, CIoU=True)
#     iou_term = (1.0 - iou) * weight

#     r = _CFG["ratio"]
#     if r <= 0.0:
#         term = iou_term
#     else:
#         if stride_fg is not None:
#             pb_px, tb_px = pb * stride_fg, tb * stride_fg
#         else:
#             pb_px, tb_px = pb, tb
#         nwd = nwd_similarity(pb_px, tb_px, _CFG["C"])
#         term = (1.0 - r) * iou_term + r * ((1.0 - nwd) * weight)

#     if not _CFG["printed"] and tb.numel() > 0:
#         _CFG["printed"] = True
#         b = tb * stride_fg if stride_fg is not None else tb
#         s = torch.sqrt((b[:, 2] - b[:, 0]).abs() * (b[:, 3] - b[:, 1]).abs())
#         m = s.mean().item()
#         print("\n" + "=" * 64)
#         print(f"[NWD] box scale sqrt(w*h) in NWD units: mean={m:.3f} "
#               f"median={s.median():.3f} min={s.min():.3f} max={s.max():.3f}")
#         print(f"[NWD] configured C={_CFG['C']}  ratio={_CFG['ratio']}")
#         print(f"[NWD] stride rescaling: {'ON (image pixels)' if stride_fg is not None else 'OFF'}")
#         if not (0.3 * m < _CFG["C"] < 3.0 * m):
#             print(f"[NWD] *** WARNING: C far from box scale. Set C near {m:.2f} ***")
#         else:
#             print("[NWD] C looks reasonable.")
#         print("=" * 64 + "\n")

#     return term.sum() / target_scores_sum


# class NWDBboxLoss(_BASE):
#     def forward(self, pred_dist, pred_bboxes, anchor_points, target_bboxes,
#                 target_scores, target_scores_sum, fg_mask,
#                 imgsz=None, stride=None):
#         weight = target_scores[fg_mask].sum(-1, keepdim=True)

#         stride_fg = None
#         if stride is not None:
#             # stride is (num_anchors, 1); fg_mask is (batch, num_anchors)
#             s = stride if stride.dim() == 3 else stride.unsqueeze(0)
#             s = s.expand(fg_mask.shape[0], -1, -1)
#             stride_fg = s[fg_mask]
#         loss_iou = _blend(pred_bboxes[fg_mask], target_bboxes[fg_mask],
#                           weight, target_scores_sum, stride_fg)

#         # --- DFL term: copied verbatim from upstream 8.4.x ---
#         if self.dfl_loss:
#             target_ltrb = bbox2dist(anchor_points, target_bboxes,
#                                     self.dfl_loss.reg_max - 1)
#             loss_dfl = self.dfl_loss(
#                 pred_dist[fg_mask].view(-1, self.dfl_loss.reg_max),
#                 target_ltrb[fg_mask]) * weight
#             loss_dfl = loss_dfl.sum() / target_scores_sum
#         else:
#             target_ltrb = bbox2dist(anchor_points, target_bboxes)
#             target_ltrb = target_ltrb * stride
#             target_ltrb[..., 0::2] /= imgsz[1]
#             target_ltrb[..., 1::2] /= imgsz[0]
#             pred_dist = pred_dist * stride
#             pred_dist[..., 0::2] /= imgsz[1]
#             pred_dist[..., 1::2] /= imgsz[0]
#             loss_dfl = F.l1_loss(pred_dist[fg_mask], target_ltrb[fg_mask],
#                                  reduction="none").mean(-1, keepdim=True) * weight
#             loss_dfl = loss_dfl.sum() / target_scores_sum

#         return loss_iou, loss_dfl


# def patch(C=18.0, ratio=0.5):
#     _CFG.update({"C": float(C), "ratio": float(ratio), "printed": False})
#     ul_loss.BboxLoss = NWDBboxLoss
#     print(f"[NWD] BboxLoss patched.  C={C}  ratio={ratio}")


# def unpatch():
#     ul_loss.BboxLoss = _BASE
#     print("[NWD] restored original BboxLoss")

In [7]:
# import sys
# sys.path.insert(0, "/kaggle/working")

# import nwd_patch
# nwd_patch.patch(C=14.66, ratio=0.0)   # ratio 0 = pure IoU

# from ultralytics import YOLO
# YOLO("yolo11n.pt").train(
#     data="/kaggle/working/data_custom.yaml",
#     epochs=30, imgsz=2048, rect=True, batch=4, device=0, cache="ram",
#     mosaic=0.0, mixup=0.0, cutmix=0.0, copy_paste=0.0, erasing=0.0,
#     scale=0.1, translate=0.05, degrees=0.0, shear=0.0, perspective=0.0,
#     flipud=0.0, fliplr=0.5, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
#     patience=100, seed=0,
#     name="sanity_ratio000", project="/kaggle/working/runs_nwd",
# )

In [8]:
# from ultralytics import YOLO

# YOLO("/kaggle/working/runs_nwd/sanity_ratio000-2/weights/best.pt").val(
#     data="/kaggle/working/data_custom.yaml",
#     imgsz=2048, rect=True, batch=1, device=0, split="val",
#     name="val_sanity_ratio000", project="/kaggle/working/runs_nwd",
# )

In [9]:
# import importlib, sys
# sys.path.insert(0, "/kaggle/working")
# import nwd_patch
# from ultralytics import YOLO

# C_REAL = 29.5     # from the [NWD] print

# for ratio in [0.0, 0.4, 0.7, 1.0]:
#     importlib.reload(nwd_patch)
#     nwd_patch.patch(C=C_REAL, ratio=ratio)

#     YOLO("yolo11n.pt").train(
#         data="/kaggle/working/data_custom.yaml",
#         epochs=150, imgsz=2048, rect=True, batch=4, device=0, cache=False,
#         mosaic=0.0, mixup=0.0, cutmix=0.0, copy_paste=0.0, erasing=0.0,
#         scale=0.1, translate=0.05, degrees=0.0, shear=0.0, perspective=0.0,
#         flipud=0.0, fliplr=0.5, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
#         patience=50, seed=0, deterministic=True,
#         verbose=False, plots=False, save_period=-1,
#         name=f"nwd_r{int(ratio*100):03d}", project="/kaggle/working/runs_nwd",
#     )

In [10]:
# import pandas as pd, glob, os

# rows = []
# for f in glob.glob("/kaggle/working/runs_nwd/**/results.csv", recursive=True):
#     df = pd.read_csv(f); df.columns = df.columns.str.strip()
#     b = df.loc[df["metrics/mAP50-95(B)"].idxmax()]
#     rows.append({"run": os.path.basename(os.path.dirname(f)),
#                  "P": round(b["metrics/precision(B)"], 4),
#                  "R": round(b["metrics/recall(B)"], 4),
#                  "mAP50": round(b["metrics/mAP50(B)"], 4),
#                  "mAP50-95": round(b["metrics/mAP50-95(B)"], 4),
#                  "epoch": int(b["epoch"])})

# print(pd.DataFrame(rows).sort_values("mAP50-95", ascending=False).to_string(index=False))

In [11]:
# from ultralytics import YOLO

# for ratio in [0.0, 0.4, 0.7, 1.0]:
#     tag = f"nwd_r{int(ratio*100):03d}"
#     YOLO(f"/kaggle/working/runs_nwd/{tag}/weights/best.pt").val(
#         data="/kaggle/working/data_custom.yaml",
#         imgsz=2048, rect=True, batch=1, device=0, split="val",
#         name=f"val_{tag}", project="/kaggle/working/runs_nwd",
#     )

In [12]:
!yolo train model=yolo11n.pt \
    data='/kaggle/working/data_custom.yaml' \
    epochs=450 \
    imgsz=16384 \
    rect=True \
    batch=1 \
    device=0 \
    cache=False \
    mosaic=0.0 \
    mixup=0.0 \
    cutmix=0.0 \
    copy_paste=0.0 \
    erasing=0.0 \
    scale=0.1 \
    translate=0.05 \
    degrees=0.0 \
    shear=0.0 \
    perspective=0.0 \
    flipud=0.0 \
    fliplr=0.5 \
    hsv_h=0.0 \
    hsv_s=0.0 \
    hsv_v=0.0 \
    patience=50 \
    seed=0 \
    deterministic=True \
    verbose=False \
    plots=False \
    name='res_4096' \
    project='/kaggle/working/runs_res'

Ultralytics 8.4.132 🚀 Python-3.11.13 torch-2.13.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_custom.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=450, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=16384, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=res_4096, nbs=64, nms=False,

In [13]:
!yolo val model='/kaggle/working/runs_res/res_4096/weights/best.pt' \
    data='/kaggle/working/data_custom.yaml' \
    imgsz=16384 \
    rect=True \
    batch=1 \
    device=0 \
    split=val \
    name='val_res_4096' \
    project='/kaggle/working/runs_res'

Traceback (most recent call last):
  File "/usr/local/bin/yolo", line 8, in <module>
    sys.exit(entrypoint())
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/ultralytics/cfg/__init__.py", line 1110, in entrypoint
    model = YOLO(model, task=task)
            ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/ultralytics/models/yolo/model.py", line 83, in __init__
    super().__init__(model=model, task=task, verbose=verbose)
  File "/usr/local/lib/python3.11/dist-packages/ultralytics/engine/model.py", line 133, in __init__
    self._load(model, task=task)
  File "/usr/local/lib/python3.11/dist-packages/ultralytics/engine/model.py", line 249, in _load
    self.model, self.ckpt = load_checkpoint(weights)
                            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/ultralytics/nn/tasks.py", line 1919, in load_checkpoint
    ckpt, weight = torch_safe_load(weight)  # load ckpt
                   ^^^^^^^^

In [14]:
import glob, os, shutil

# drop last.pt, keep best.pt
for f in glob.glob("/kaggle/working/runs_*/**/last.pt", recursive=True):
    os.remove(f)

# drop training plots and sample batches
for pat in ["*.jpg", "*.png", "events.out.tfevents*"]:
    for f in glob.glob(f"/kaggle/working/runs_*/**/{pat}", recursive=True):
        os.remove(f)

# ultralytics clone, if you still have it
shutil.rmtree("/kaggle/working/ultralytics", ignore_errors=True)

# what's left
tot = sum(os.path.getsize(f) for f in glob.glob("/kaggle/working/**", recursive=True) if os.path.isfile(f))
print(f"{tot/1e6:.1f} MB remaining")

11.3 MB remaining
